# Iris classification | Mage block pipeline
Parts C1–C4. Classroom split: 80/20, seed 42, decision tree, default depth 3, accuracy threshold 0.8. The local model cells execute the actual functions with identity decorators; live Mage scheduling and browser screenshots are separate, opt-in steps. This notebook includes the project sources for standalone use and also matches the supplied Iris project archive.

In [1]:
RUN_MAGE = False
# Install Python 3.12-compatible dependencies in your own environment before enabling Mage:
# python -m pip install mage-ai scikit-learn pandas joblib pytest pyyaml
from pathlib import Path
import sys, tempfile, os, json, re, subprocess, shutil
import pandas as pd
from IPython.display import display
WORKSPACE = Path.cwd() if RUN_MAGE else Path(tempfile.mkdtemp(prefix='iris_local_'))

## Bootstrap source files
Existing differing files are not overwritten. Use a clean directory or inspect and resolve differences before running.

In [2]:
PROJECT_FILES = {'iris_project/baseline_metadata.yaml': 'blocks:\n- name: load_iris\n  uuid: load_iris\n  type: data_loader\n  language: python\n  upstream_blocks: []\n  downstream_blocks:\n  - prepare_data\n- name: prepare_data\n  uuid: prepare_data\n  type: transformer\n  language: python\n  upstream_blocks:\n  - load_iris\n  downstream_blocks:\n  - train_model\n- name: train_model\n  uuid: train_model\n  type: transformer\n  language: python\n  upstream_blocks:\n  - prepare_data\n  downstream_blocks:\n  - save_results\n- name: save_results\n  uuid: save_results\n  type: data_exporter\n  language: python\n  upstream_blocks:\n  - train_model\n  downstream_blocks: []\nname: iris_classifier\nuuid: iris_classifier\ntype: python\nvariables:\n  max_depth: 3\n', 'iris_project/metadata.yaml': 'project_type: standalone\nvariables_dir: .mage_data\n', 'iris_project/local_check.py': '"""Local direct function calls, not execution by the Mage scheduler."""\nfrom pathlib import Path\nfrom tempfile import TemporaryDirectory\nimport os\nimport pandas as pd\nfrom sklearn.metrics import confusion_matrix\n\n\ndef run_checks():\n    # Inject identity decorators for this explicitly local run, even if Mage is installed.\n    def block(relative_path):\n        path = Path(__file__).resolve().parent / relative_path\n        def identity(fn): return fn\n        ns = {key: identity for key in (\'data_loader\',\'transformer\',\'data_exporter\',\'test\')}\n        exec(compile(path.read_text(encoding=\'utf8\'), str(path), \'exec\'), ns)\n        return ns\n    load = block(\'data_loaders/load_iris.py\')\n    validate = block(\'transformers/validate_data.py\')\n    prepare = block(\'transformers/prepare_data.py\')\n    train = block(\'transformers/train_model.py\')\n    evaluate = block(\'transformers/evaluate_model.py\')\n    save = block(\'data_exporters/save_results.py\')\n    metrics_save = block(\'data_exporters/save_metrics.py\')\n    raw = load[\'load_data\'](); load[\'test_output\'](raw)\n    clean = validate[\'validate\'](raw); validate[\'test_valid\'](clean)\n    assert clean.equals(raw)\n    split = prepare[\'transform\'](clean); prepare[\'test_output\'](split)\n    rows = []\n    previous = Path.cwd()\n    with TemporaryDirectory() as temp:\n        try:\n            os.chdir(temp)\n            for depth in range(1, 6):\n                for filename in (\'iris_predictions.csv\', \'iris_metrics.csv\'):\n                    Path(filename).unlink(missing_ok=True)\n                pred = train[\'train\'](split, max_depth=depth)\n                accuracy = (pred.target == pred.predicted).mean()\n                try:\n                    train[\'test_accuracy\'](pred)\n                except AssertionError:\n                    passed = False\n                else:\n                    passed = True\n                    save[\'test_saved\'](save[\'export_data\'](pred))\n                    metrics = evaluate[\'evaluate\'](pred)\n                    evaluate[\'test_metrics\'](metrics)\n                    metrics_save[\'test_metric_file\'](metrics_save[\'export_metrics\'](metrics))\n                assert Path(\'iris_predictions.csv\').exists() == passed\n                assert Path(\'iris_metrics.csv\').exists() == passed\n                rows.append({\'max_depth\': depth, \'accuracy\': float(accuracy), \'local_gate\': \'PASS\' if passed else \'FAIL\', \'predictions_csv\': passed, \'metrics_csv\': passed})\n                if depth == 3:\n                    assert passed\n                    matrix = confusion_matrix(pred.target, pred.predicted, labels=[0, 1, 2])\n                    default_metrics = metrics.copy()\n        finally:\n            os.chdir(previous)\n    return pd.DataFrame(rows), pd.DataFrame(matrix, index=pd.Index([0,1,2], name=\'actual\'), columns=pd.Index([0,1,2], name=\'predicted\')), default_metrics\n\nif __name__ == \'__main__\':\n    for item in run_checks(): print(item.to_string())\n', 'iris_project/__init__.py': '', 'iris_project/pipelines/__init__.py': '', 'iris_project/data_loaders/load_iris.py': 'from sklearn.datasets import load_iris\n\ntry:\n    data_loader\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import data_loader\n    except ModuleNotFoundError as exc:\n        if exc.name != \'mage_ai\':\n            raise\n        # Local unit tests do not require the Mage scheduler.\n        def data_loader(fn):\n            return fn\ntry:\n    test\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import test\n    except ModuleNotFoundError as exc:\n        if exc.name != \'mage_ai\':\n            raise\n        def test(fn):\n            return fn\n\n@data_loader\ndef load_data(*args, **kwargs):\n    frame = load_iris(as_frame=True).frame\n    print(f"Iris rows: {len(frame)}")\n    return frame\n\n@test\ndef test_output(output, *args):\n    assert output.shape == (150, 5)\n    assert set(output[\'target\']) == {0, 1, 2}\n', 'iris_project/data_loaders/__init__.py': '', 'iris_project/transformers/prepare_data.py': 'import pandas as pd\nfrom sklearn.model_selection import train_test_split\n\ntry:\n    transformer\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import transformer\n    except ModuleNotFoundError as exc:\n        if exc.name != \'mage_ai\':\n            raise\n        # Local unit tests do not require the Mage scheduler.\n        def transformer(fn):\n            return fn\ntry:\n    test\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import test\n    except ModuleNotFoundError as exc:\n        if exc.name != \'mage_ai\':\n            raise\n        def test(fn):\n            return fn\n\n@transformer\ndef transform(df, *args, **kwargs):\n    # Keep the lecture split and its row order (no stratification).\n    train, holdout = train_test_split(df, test_size=0.2, random_state=42)\n    result = pd.concat([train.assign(split=\'train\'), holdout.assign(split=\'test\')])\n    print(f"Train rows: {len(train)} | Test rows: {len(holdout)}")\n    return result\n\n@test\ndef test_output(output, *args):\n    assert len(output) == 150\n    assert (output[\'split\'] == \'train\').sum() == 120\n    assert (output[\'split\'] == \'test\').sum() == 30\n    assert output.index.is_unique\n', 'iris_project/transformers/train_model.py': "import joblib\nfrom sklearn.tree import DecisionTreeClassifier\nfrom sklearn.metrics import accuracy_score\n\ntry:\n    transformer\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import transformer\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        # Local unit tests do not require the Mage scheduler.\n        def transformer(fn):\n            return fn\ntry:\n    test\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import test\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        def test(fn):\n            return fn\n\nFEATURES = ['sepal length (cm)', 'sepal width (cm)',\n            'petal length (cm)', 'petal width (cm)']\n\n@transformer\ndef train(df, *args, **kwargs):\n    depth = int(kwargs.get('max_depth', 3))\n    fitting = df.loc[df.split == 'train']\n    holdout = df.loc[df.split == 'test'].copy()\n    model = DecisionTreeClassifier(max_depth=depth, random_state=42)\n    model.fit(fitting[FEATURES], fitting.target)\n    holdout['predicted'] = model.predict(holdout[FEATURES])\n    score = accuracy_score(holdout.target, holdout.predicted)\n    print(f'Max depth: {depth} | Test accuracy = {score:.6f}')\n    # The pickle is a training artifact; the prediction CSV is gated downstream.\n    joblib.dump(model, 'iris_model.pkl')\n    return holdout\n\n@test\ndef test_accuracy(output, *args):\n    assert len(output) == 30\n    assert accuracy_score(output.target, output.predicted) >= 0.8, 'Model accuracy is below 80%'\n", 'iris_project/transformers/evaluate_model.py': "import pandas as pd\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix\n\ntry:\n    transformer\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import transformer\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        # Local unit tests do not require the Mage scheduler.\n        def transformer(fn):\n            return fn\ntry:\n    test\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import test\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        def test(fn):\n            return fn\n\n@transformer\ndef evaluate(df, *args, **kwargs):\n    actual, predicted = df.target, df.predicted\n    labels = [0, 1, 2]\n    matrix = pd.DataFrame(confusion_matrix(actual, predicted, labels=labels),\n                          index=pd.Index(labels, name='actual'),\n                          columns=pd.Index(labels, name='predicted'))\n    print('Confusion matrix:')\n    print(matrix.to_string())\n    macro = dict(average='macro', labels=labels, zero_division=0)\n    metrics = pd.DataFrame({'metric': ['accuracy', 'precision', 'recall', 'f1_score'],\n                            'value': [accuracy_score(actual, predicted),\n                                      precision_score(actual, predicted, **macro),\n                                      recall_score(actual, predicted, **macro),\n                                      f1_score(actual, predicted, **macro)]})\n    print(metrics.to_string(index=False))\n    return metrics\n\n@test\ndef test_metrics(output, *args):\n    assert list(output.metric) == ['accuracy', 'precision', 'recall', 'f1_score']\n    assert output.value.between(0, 1).all()\n", 'iris_project/transformers/validate_data.py': "try:\n    transformer\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import transformer\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        # Local unit tests do not require the Mage scheduler.\n        def transformer(fn):\n            return fn\ntry:\n    test\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import test\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        def test(fn):\n            return fn\n\n@transformer\ndef validate(df, *args, **kwargs):\n    return df.copy()\n\n@test\ndef test_valid(output, *args):\n    assert not output.isna().any().any(), 'Missing measurement or target'\n    assert set(output['target']) == {0, 1, 2}, 'Expected all three Iris classes'\n    assert output.shape == (150, 5)\n", 'iris_project/transformers/__init__.py': '', 'iris_project/data_exporters/save_metrics.py': "from pathlib import Path\nimport pandas as pd\n\ntry:\n    data_exporter\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import data_exporter\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        # Local unit tests do not require the Mage scheduler.\n        def data_exporter(fn):\n            return fn\ntry:\n    test\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import test\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        def test(fn):\n            return fn\n\n@data_exporter\ndef export_metrics(df, *args, **kwargs):\n    path = Path('iris_metrics.csv')\n    tmp = path.with_suffix('.tmp')\n    df.to_csv(tmp, index=False)\n    tmp.replace(path)\n    print(f'Metrics exported: {len(df)} rows')\n    return df\n\n@test\ndef test_metric_file(output, *args):\n    persisted = pd.read_csv('iris_metrics.csv')\n    assert len(persisted) == len(output) == 4\n    assert persisted.metric.tolist() == output.metric.tolist()\n    assert persisted.value.between(0, 1).all()\n", 'iris_project/data_exporters/save_results.py': "from pathlib import Path\nimport pandas as pd\n\ntry:\n    data_exporter\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import data_exporter\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        # Local unit tests do not require the Mage scheduler.\n        def data_exporter(fn):\n            return fn\ntry:\n    test\nexcept NameError:\n    try:\n        from mage_ai.data_preparation.decorators import test\n    except ModuleNotFoundError as exc:\n        if exc.name != 'mage_ai':\n            raise\n        def test(fn):\n            return fn\n\n@data_exporter\ndef export_data(df, *args, **kwargs):\n    dest = Path('iris_predictions.csv')\n    pending = dest.with_suffix('.tmp')\n    df.to_csv(pending, index=False)\n    pending.replace(dest)\n    print(f'Saved {len(df)} rows to {dest}')\n    return df\n\n@test\ndef test_saved(output, *args):\n    written = pd.read_csv('iris_predictions.csv')\n    assert len(written) == len(output) == 30\n    assert {'target', 'predicted'}.issubset(written.columns)\n    assert written.predicted.tolist() == output.predicted.tolist()\n", 'iris_project/data_exporters/__init__.py': '', 'iris_project/pipelines/iris_classifier/metadata.yaml': 'blocks:\n- name: load_iris\n  uuid: load_iris\n  type: data_loader\n  language: python\n  upstream_blocks: []\n  downstream_blocks:\n  - validate_data\n- name: validate_data\n  uuid: validate_data\n  type: transformer\n  language: python\n  upstream_blocks:\n  - load_iris\n  downstream_blocks:\n  - prepare_data\n- name: prepare_data\n  uuid: prepare_data\n  type: transformer\n  language: python\n  upstream_blocks:\n  - validate_data\n  downstream_blocks:\n  - train_model\n- name: train_model\n  uuid: train_model\n  type: transformer\n  language: python\n  upstream_blocks:\n  - prepare_data\n  downstream_blocks:\n  - save_results\n  - evaluate_model\n- name: save_results\n  uuid: save_results\n  type: data_exporter\n  language: python\n  upstream_blocks:\n  - train_model\n  downstream_blocks: []\n- name: evaluate_model\n  uuid: evaluate_model\n  type: transformer\n  language: python\n  upstream_blocks:\n  - train_model\n  downstream_blocks:\n  - save_metrics\n- name: save_metrics\n  uuid: save_metrics\n  type: data_exporter\n  language: python\n  upstream_blocks:\n  - evaluate_model\n  downstream_blocks: []\nname: iris_classifier\nuuid: iris_classifier\ntype: python\nvariables:\n  max_depth: 3\n', 'iris_project/pipelines/iris_classifier/__init__.py': ''}
for relative, source in PROJECT_FILES.items():
    target=WORKSPACE/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.exists() and target.read_text(encoding='utf8')!=source:
        raise RuntimeError(f'Existing source differs: {relative}; review before replacing')
    target.write_text(source,encoding='utf8')
sys.path.insert(0,str(WORKSPACE))

## C1–C4 — local direct-call check

In [3]:
from iris_project.local_check import run_checks
results,matrix,metrics=run_checks()
print('DIRECT CALLS ONLY — not Mage runtime')
display(results.round(4))
display(matrix)
display(metrics.round(4))

Iris rows: 150
Train rows: 120 | Test rows: 30
Max depth: 1 | Test accuracy = 0.633333
Max depth: 2 | Test accuracy = 0.966667
Saved 30 rows to iris_predictions.csv
Confusion matrix:
predicted   0  1   2
actual              
0          10  0   0
1           0  8   1
2           0  0  11
   metric    value
 accuracy 0.966667
precision 0.972222
   recall 0.962963
 f1_score 0.965899
Metrics exported: 4 rows
Max depth: 3 | Test accuracy = 1.000000
Saved 30 rows to iris_predictions.csv
Confusion matrix:
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score    1.0
Metrics exported: 4 rows
Max depth: 4 | Test accuracy = 1.000000
Saved 30 rows to iris_predictions.csv
Confusion matrix:
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score   

,max_depth,accuracy,local_gate,predictions_csv,metrics_csv
0,1,0.6333,FAIL,False,False
1,2,0.9667,PASS,True,True
2,3,1.0000,PASS,True,True
3,4,1.0000,PASS,True,True
4,5,1.0000,PASS,True,True


predicted,0,1,2
actual,,,
0,10,0,0
1,0,9,0
2,0,0,11


,metric,value
0,accuracy,1.0
1,precision,1.0
2,recall,1.0
3,f1_score,1.0


## C1 — actual Mage baseline
In Colab use a compatible Mage environment, for example a Python 3.12 virtual environment, and set `MAGE_BIN` explicitly if the command is not on the kernel PATH. Run in a dedicated workspace; do not use an unrelated `iris_predictions.csv`. The baseline graph is temporarily installed and restored even if an experiment fails.

In [4]:
MAGE_BIN = os.environ.get('MAGE_BIN') or shutil.which('mage') or 'mage'
metadata=WORKSPACE/'iris_project/pipelines/iris_classifier/metadata.yaml'
baseline_meta=WORKSPACE/'iris_project/baseline_metadata.yaml'
logs=WORKSPACE/'evidence_mage'

def run_mage(depth, label):
    logs.mkdir(exist_ok=True)
    for f in ('iris_predictions.csv','iris_metrics.csv'):
        (WORKSPACE/f).unlink(missing_ok=True)
    cmd=[MAGE_BIN,'run','iris_project','iris_classifier','--test','--runtime-vars',json.dumps({'max_depth':depth})]
    proc=subprocess.run(cmd,cwd=WORKSPACE,capture_output=True,text=True,timeout=300)
    text=proc.stdout+'\n'+proc.stderr
    (logs/f'{label}.log').write_text(text,encoding='utf8')
    print(text)
    match=re.findall(r'Test accuracy\s*=\s*([0-9.]+)',text)
    score=float(match[-1]) if match else None
    created=(WORKSPACE/'iris_predictions.csv').exists()
    # Distinguish a genuine accuracy rejection from missing CLI/dependency errors.
    if score is None: status='ERROR (accuracy unavailable)'
    elif score < 0.8 and not created and ('Model accuracy is below 80%' in text or 'test_accuracy' in text): status='FAIL'
    elif score >= 0.8 and proc.returncode==0 and created: status='PASS'
    else: status='UNVERIFIED'
    return dict(max_depth=depth,accuracy=score,test_status=status,csv_created=created,exit_code=proc.returncode)

if RUN_MAGE:
    print(subprocess.run([MAGE_BIN,'run','--help'],capture_output=True,text=True).stdout)
    original=metadata.read_text()
    try:
        metadata.write_text(baseline_meta.read_text())
        baseline_result=run_mage(3,'C1_baseline')
        display(pd.DataFrame([baseline_result]))
        assert baseline_result['test_status']=='PASS'
    finally:
        metadata.write_text(original)
else:
    print('Mage CLI not run in this local notebook output.')

Mage CLI not run in this local notebook output.


## C2 — depth experiments in Mage
Run the four baseline blocks for each depth. The prediction CSV is removed before each run. A model pickle can still exist after a failing training test; the prediction export should not.

In [5]:
if RUN_MAGE:
    original=metadata.read_text()
    try:
        metadata.write_text(baseline_meta.read_text())
        observed=[run_mage(depth,f'C2_depth_{depth}') for depth in range(1,6)]
        display(pd.DataFrame(observed))
        assert all(r['test_status'] in {'PASS','FAIL'} for r in observed)
        assert any(r['test_status']=='FAIL' for r in observed)
        assert all(r['csv_created']==(r['test_status']=='PASS') for r in observed)
    finally:
        metadata.write_text(original)

## C3 and C4 — final graph, validation and metrics export
Final graph adds an input validator before preparation and a metrics exporter after evaluation. `train_model` still has exactly two direct downstream blocks. The additional exporter is useful evidence, not a separate guaranteed bonus.

In [6]:
print(metadata.read_text(encoding='utf8'))
if RUN_MAGE:
    final=run_mage(3,'C3_C4_final')
    assert final['test_status']=='PASS'
    predictions=pd.read_csv(WORKSPACE/'iris_predictions.csv')
    actual_metrics=pd.read_csv(WORKSPACE/'iris_metrics.csv')
    display(pd.crosstab(predictions.target,predictions.predicted,rownames=['actual'],colnames=['predicted']))
    display(actual_metrics)
    assert len(predictions)==30 and len(actual_metrics)==4

blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks:
  - validate_data
- name: validate_data
  uuid: validate_data
  type: transformer
  language: python
  upstream_blocks:
  - load_iris
  downstream_blocks:
  - prepare_data
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks:
  - validate_data
  downstream_blocks:
  - train_model
- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks:
  - prepare_data
  downstream_blocks:
  - save_results
  - evaluate_model
- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks:
  - train_model
  downstream_blocks: []
- name: evaluate_model
  uuid: evaluate_model
  type: transformer
  language: python
  upstream_blocks:
  - train_model
  downstream_blocks:
  - save_metrics
- name: save_metrics
  uuid: save_metrics
  type: data_exporter
  language:

## Mage UI evidence
In your authenticated environment start `mage start iris_project`, open the local UI on port 6789, select `iris_classifier`, and capture the real DAG. In Colab, use the instructor-approved Cloudflare tunnel instructions, do not expose credentials, and stop the tunnel immediately afterwards. The notebook does not fabricate UI screenshots.

**CI:** the included archive contains the assignment pytest test and a workflow for Python 3.12. Run pytest, demonstrate the 0.3 failure as a separate genuine Git commit, restore 0.2, and capture the actual Actions results.